# MATH Level 5 — competition maths (local)

MATH Level 5 (`leaderboard_math_hard`) is the hardest tier of competition maths problems — 1,324 problems, 4-shot, generative. The model must produce the correct final answer.

This notebook runs the benchmark **locally**, straight against the model's chat
endpoint — no EvalHub and no OpenShift. It is self-contained: all settings are in the
configuration cell below, and no job file is needed. It is the notebook equivalent of
the matching `run-*-local.sh` script.

**How it works:** [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
(`lm_eval`) is run through `uvx`, which fetches it on demand into a throwaway
environment. Questions are sent to the model's `/chat/completions` endpoint with the
model's chat template applied, and answers are scored locally.

**Requires:** [`uv`](https://docs.astral.sh/uv/) (for `uvx`) and network access to the
model endpoint.

## 1. Configuration

Everything you might want to change lives in this one cell.

In [ ]:
import json, os, shutil, subprocess
from getpass import getpass
from pathlib import Path

# --- Model endpoint (from .env, or prompted for if there is no .env) ---
def load_env(path=".env"):
    """Read KEY=VALUE lines from a .env file into os.environ. Returns False if there is no file."""
    p = Path(path)
    if not p.is_file():
        return False
    for line in p.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip("\"'")
    return True

if load_env():
    print("Loaded settings from .env")
    missing = [k for k in ("OPENAI_BASE_URL", "OPENAI_MODEL") if not os.environ.get(k)]
    if missing:
        raise SystemExit(f"Please set {', '.join(missing)} in .env (see .env.template)")
else:
    print("No .env file found (copy .env.template to .env to skip these prompts)")
    os.environ["OPENAI_BASE_URL"] = input("Model base URL (e.g. https://host/v1): ").strip()
    os.environ["OPENAI_MODEL"] = input("Model name: ").strip()
    os.environ["OPENAI_API_KEY"] = getpass("API key (leave blank if none): ")

MODEL_URL = os.environ["OPENAI_BASE_URL"].rstrip("/")  # OpenAI-compatible base URL
MODEL_NAME = os.environ["OPENAI_MODEL"]
MODEL_API_KEY = os.environ.get("OPENAI_API_KEY", "")


# --- Benchmark ---
TASK = "leaderboard_math_hard"  # lm-eval task name
LM_EVAL_VERSION = "0.4.12"  # pinned so results are repeatable
LM_EVAL_EXTRAS = "api,math"  # pip extras lm-eval needs for this task
NUM_CONCURRENT = 8          # parallel requests to the model
THRESHOLD = 0.1  # minimum score to count as PASS
LIMIT = None                # e.g. 10 for a quick smoke test; None runs the full benchmark


## Helper

`run()` launches a command and streams its output into the notebook as it arrives, so you
can watch progress. It raises an error if the command exits non-zero.

In [ ]:
def run(cmd, env=None):
    proc = subprocess.Popen(cmd, env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{cmd[0]} exited with {proc.returncode}")

## 2. Run the benchmark

Builds the `lm_eval` command and runs it. Completed requests are cached on disk, so if
the run dies partway (for example the gateway drops a connection), re-running this cell
resumes where it left off. The cache is deleted after a successful run so the next run
always gets fresh answers from the model.

In [ ]:
OUTPUT = Path(f".eval-notebook-{TASK}")             # lm-eval writes results and samples here
CACHE = Path(f".eval-notebook-{TASK}-cache") / "requests"
CACHE.parent.mkdir(exist_ok=True)
shutil.rmtree(OUTPUT, ignore_errors=True)

cmd = ["uvx", "--python", "3.12", "--from", f"lm-eval[{LM_EVAL_EXTRAS}]=={LM_EVAL_VERSION}",
       "lm_eval", "--model", "local-chat-completions",
       "--model_args", f"model={MODEL_NAME},base_url={MODEL_URL.rstrip('/')}/chat/completions,"
                       f"num_concurrent={NUM_CONCURRENT},max_retries=5,tokenized_requests=False",
       "--tasks", TASK, "--apply_chat_template", "--log_samples",
       "--output_path", str(OUTPUT), "--use_cache", str(CACHE)]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]

run(cmd, env={"OPENAI_API_KEY": MODEL_API_KEY or "not-required"})
shutil.rmtree(CACHE.parent, ignore_errors=True)

## 3. Score and pass/fail

The score is `exact_match` on the final answer, averaged over the maths subjects. Scores are low even for strong models, hence the low threshold. Per-question model answers are in the `.eval-notebook-<task>` folder (`samples_*.jsonl`) if you want to see what it got wrong.

In [ ]:
results_file = next(OUTPUT.rglob("results_*.json"))
scores = json.loads(results_file.read_text())["results"][TASK]

# The group-level exact_match is the overall score across all subjects.
score = next(v for k, v in scores.items() if k.startswith("exact_match,"))
print(f"exact_match: {score:.3f}")
print("PASS" if score >= THRESHOLD else "FAIL", f"- exact_match {score:.3f} vs threshold {THRESHOLD}")